### Notebook scope

### Purpose
Assess BWCN reference O3 severity for selected hindcast years.

### Scientific question
Is 0008 the most severe event and therefore the best detailed source-diagnosis case?

### Method
Use BWCN partial O3 60-90N, 30-70 hPa for years 0003, 0008, 0013, 0014, 0019. Compute Jan-May time series and March-April minimum.

### Expected output
One CSV and one two-panel figure.

### Interpretation
If 0008 has the lowest March-April minimum, it supports treating 0008-01 as the detailed source-mechanism case.

### Caveat
0003 uses BWCN .002 reference year 0003 while hindcast 0003 is a special branch; report that caveat.


### Setup imports and shared paths

### Purpose
Load the shared Extention_analysis utility module and create output directories.

### Scientific question
Can every notebook start from a clean kernel and still find the same data roots and output roots?

### Method
Use DATA_ROOT, HINDCAST_ROOT, BWCN_ROOT, B2000_ROOT, WORK_ROOT, FIG_DIR, TAB_DIR, CACHE_DIR, and LOG_DIR from hindcast_extension_utils.py. No diagnostics are calculated in this cell.

### Expected output
Printed path check only. No figure is generated by this setup cell.

### Interpretation
Successful execution means the notebook can access the common utilities and write outputs in the agreed directory tree.

### Caveat
This setup does not prove that every downstream data product exists; missing products are logged later.


In [ ]:
from pathlib import Path
import sys

WORK_ROOT = Path("/home/weiji/restart_exam/code_cleaned/Hindcast_experiment/Extention_analysis")
if str(WORK_ROOT) not in sys.path:
    sys.path.insert(0, str(WORK_ROOT))

import math
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter

import hindcast_extension_utils as hx
from hindcast_extension_utils import *
_assign_member_short = hx._assign_member_short

for d in [FIG_DIR, TAB_DIR, CACHE_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("WORK_ROOT", WORK_ROOT)
print("HINDCAST_ROOT", HINDCAST_ROOT)

### Plot reference O3 severity

### Purpose
Generate reference O3 time series and severity bar plot.

### Scientific question
How extreme are the selected reference years relative to each other?

### Method
For each selected year, load BWCN reference O3; x-axis is DOY Jan1-May30. March-April minimum is computed over DOY 60-120.

### Expected output
outputs/tables/01_reference_O3_severity_selected_years.csv and outputs/figures/01_reference_O3_severity_selected_years.png/pdf.

### Interpretation
Lower March-April minimum means stronger ozone depletion. Highlighting 0008 makes it easy to compare event severity.

### Caveat
This is a reference-state comparison only; it does not use hindcast ensemble spread.


In [ ]:
rows = []
series_rows = []
for year in SELECTED_YEARS:
    da, date = load_bwcn_reference_o3(year)
    if da is None:
        continue
    mask = date_mask_from_mmdd_or_leadday(date, (1, 1), (5, 30))
    sub = da.isel(lead_time=mask)
    doys = date_to_doy(date[mask])
    for doy, val in zip(doys, sub.values):
        series_rows.append({"year": year, "doy": int(doy), "O3_DU": float(val)})
    ma = o3_ma_min(da, date)
    rows.append({"year": year, "MA_min_O3_DU": float(ma["O3_MA_min"].iloc[0]) if not ma.empty else np.nan, "special_caveat": year == "0003"})
severity = pd.DataFrame(rows)
series = pd.DataFrame(series_rows)
out_csv = TAB_DIR / "01_reference_O3_severity_selected_years.csv"
severity.merge(series.groupby("year")["O3_DU"].mean().rename("JanMay_mean_O3_DU"), on="year", how="left").to_csv(out_csv, index=False)
series.to_csv(TAB_DIR / "01_reference_O3_timeseries_selected_years.csv", index=False)
fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.2), constrained_layout=True)
for year, sub in series.groupby("year"):
    lw = 2.8 if year == "0008" else 1.7
    color = "crimson" if year == "0008" else None
    axes[0].plot(sub["doy"], sub["O3_DU"], label=year, lw=lw, color=color)
axes[0].set_xlabel("Day of year")
axes[0].set_ylabel("Partial O3, 60-90N, 30-70 hPa (DU)")
axes[0].set_title("BWCN reference O3 evolution")
axes[0].legend(ncol=2)
colors = ["crimson" if y == "0008" else "0.55" for y in severity["year"]]
axes[1].bar(severity["year"], severity["MA_min_O3_DU"], color=colors, edgecolor="black")
axes[1].set_ylabel("March-April minimum O3 (DU)")
axes[1].set_title("Reference event severity")
savefig(fig, "01_reference_O3_severity_selected_years", notebook="01_reference_event_severity.ipynb", scientific_question="Is 0008 the most severe selected reference event?", variables_windows="BWCN reference O3; 60-90N; 30-70 hPa; Jan1-May30 and March-April minimum", interpretation="The lowest bar marks the strongest ozone-depletion reference event; 0008 should stand out if it is the detailed source case.", caveat="0003 is a special hindcast branch; the BWCN .002 reference is used only as a common baseline.", csv_table=out_csv)
plt.show()
write_figure_guide()